# Prediksi Geser Mingguan dengan Model Final (Tanpa Retraining)

Semua model final di-_load_ dari disk, lalu prediksi dijalankan dari origin 3, 10, 17, 24, dan 31 Agustus 2026. Model **tidak dilatih ulang**: yang digeser hanya `last_window` (riwayat harga sampai tanggal origin). Prediksi dibandingkan dengan `data_aktual_cleaned.csv`.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from skforecast.utils import load_forecaster
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    root_mean_squared_error,
)

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_rows", 200)

MODEL_DIRS = {
    "LightGBM (default)": "final_lgbm_default",
    "LightGBM (GA)": "final_lgbm",
    "XGBoost (default)": "final_xgb_default",
    "XGBoost (GA)": "final_xgb",
}
HORIZONS = [1, 7, 30]
ORIGINS = pd.to_datetime(
    ["2026-08-03", "2026-08-10", "2026-08-17", "2026-08-24", "2026-08-31"]
)

## Cek Rentang Dataset

Origin terakhir + horizon terpanjang harus masih tercakup data aktual.


In [ ]:
df_dataset = pd.read_csv("dataset.csv", index_col="tanggal", parse_dates=True)
df_aktual = pd.read_csv(
    "data_aktual_cleaned.csv", index_col="tanggal", parse_dates=True
).asfreq("D")
commodity_cols = df_aktual.columns.tolist()

last_train = df_dataset.dropna(how="all").index.max()
butuh_sampai = ORIGINS.max() + pd.Timedelta(days=max(HORIZONS))
print("Data latih model final s.d.  :", last_train.date())
print(
    "Data aktual cleaned           :",
    df_aktual.index.min().date(),
    "->",
    df_aktual.index.max().date(),
)
print("Dibutuhkan s.d. (31 Agt + 30) :", butuh_sampai.date())
print("Missing di data aktual        :", int(df_aktual.isna().sum().sum()))

assert df_aktual.index.min() == last_train + pd.Timedelta(days=1), (
    "data aktual harus lanjut tepat setelah data latih"
)
assert df_aktual.index.max() >= butuh_sampai, (
    "data aktual tidak cukup untuk H+30 dari origin terakhir"
)
assert not df_aktual.isna().any().any()

Data latih model final s.d.  : 2026-08-03
Data aktual cleaned           : 2026-08-04 -> 2026-10-02
Dibutuhkan s.d. (31 Agt + 30) : 2026-09-30
Missing di data aktual        : 0


## Load Semua Model Final


In [3]:
forecasters = {
    (model, col, h): load_forecaster(
        str(Path(folder) / f"{folder}_{col}_h{h}.joblib"), verbose=False
    )
    for model, folder in MODEL_DIRS.items()
    for col in commodity_cols
    for h in HORIZONS
}
print(len(forecasters), "model ter-load")

108 model ter-load


## Prediksi Geser Mingguan

Riwayat skala log = `last_window_` bawaan model (berakhir 3 Agt) + `log1p(data aktual)`. Untuk tiap origin, `last_window` dipotong sampai tanggal origin, lalu `predict` dipanggil. Tidak ada `fit`.


In [ ]:
rows = []
pred_rows = []

for (model, col, h), forecaster in forecasters.items():
    y_log = pd.concat(
        [forecaster.last_window_.iloc[:, 0], np.log1p(df_aktual[col])]
    ).asfreq("D")
    assert not y_log.isna().any(), f"riwayat bolong: {model} {col} h{h}"

    # Sanity check: origin 3 Agt dengan last_window harus sama persis dengan prediksi bawaan model.
    assert np.allclose(
        forecaster.predict(steps=h),
        forecaster.predict(steps=h, last_window=y_log.loc[: ORIGINS[0]]),
    )

    for origin in ORIGINS:
        pred = np.expm1(forecaster.predict(steps=h, last_window=y_log.loc[:origin]))
        aktual = df_aktual[col].reindex(pred.index)
        rows.append(
            {
                "Model": model,
                "Komoditas": col,
                "Horizon": h,
                "Origin": origin.date(),
                "Periode Prediksi": f"{pred.index[0]:%d %b} - {pred.index[-1]:%d %b}",
                "MAE": mean_absolute_error(aktual, pred),
                "RMSE": root_mean_squared_error(aktual, pred),
                "MAPE (%)": mean_absolute_percentage_error(aktual, pred) * 100,
            }
        )
        pred_rows.extend(
            {
                "Model": model,
                "Komoditas": col,
                "Horizon": h,
                "Origin": origin.date(),
                "Tanggal": t.date(),
                "Harga Aktual": a,
                "Harga Prediksi": p,
            }
            for t, a, p in zip(pred.index, aktual, pred)
        )

df_error = pd.DataFrame(rows)
df_pred = pd.DataFrame(pred_rows)
df_error

,Model,Komoditas,Horizon,Origin,Periode Prediksi,MAE,RMSE,MAPE (%)
0,LightGBM (default),Beras Kualitas Bawah I,1,2026-08-03,04 Aug - 04 Aug,0.0015,0.0015,0.0114
1,LightGBM (default),Beras Kualitas Bawah I,1,2026-08-10,11 Aug - 11 Aug,0.0027,0.0027,0.0202
2,LightGBM (default),Beras Kualitas Bawah I,1,2026-08-17,18 Aug - 18 Aug,0.0176,0.0176,0.1325
3,LightGBM (default),Beras Kualitas Bawah I,1,2026-08-24,25 Aug - 25 Aug,0.0106,0.0106,0.0796
4,LightGBM (default),Beras Kualitas Bawah I,1,2026-08-31,01 Sep - 01 Sep,0.0396,0.0396,0.2900
...,...,...,...,...,...,...,...,...
535,XGBoost (GA),Cabai Rawit Merah,30,2026-08-03,04 Aug - 02 Sep,5.2272,6.7108,9.5557
536,XGBoost (GA),Cabai Rawit Merah,30,2026-08-10,11 Aug - 09 Sep,6.7102,8.9354,11.1439
537,XGBoost (GA),Cabai Rawit Merah,30,2026-08-17,18 Aug - 16 Sep,11.2451,12.8615,17.9466
538,XGBoost (GA),Cabai Rawit Merah,30,2026-08-24,25 Aug - 23 Sep,9.9447,11.1624,14.9941


## Tabel MAPE (%) per Origin

Baris = model × komoditas × horizon, kolom = tanggal origin.


In [5]:
tabel_mape = df_error.pivot_table(
    index=["Komoditas", "Horizon", "Model"], columns="Origin", values="MAPE (%)"
)
tabel_mape["Rata-rata"] = tabel_mape.mean(axis=1)
tabel_mape.style.format("{:.2f}").background_gradient(cmap="RdYlGn_r", axis=None)

## Ringkasan: Rata-rata Error per Model dan Horizon


In [ ]:
ringkasan = (
    df_error.groupby(["Horizon", "Model"])[["MAE", "RMSE", "MAPE (%)"]]
    .mean()
    .sort_values(["Horizon", "MAPE (%)"])
)
ringkasan["Peringkat MAPE"] = (
    ringkasan.groupby("Horizon")["MAPE (%)"].rank(method="min").astype(int)
)
ringkasan

MAE   RMSE  MAPE (%)  Peringkat MAPE
Horizon Model                                                     
1       XGBoost (GA)       0.1597 0.1597    0.3320               1
        LightGBM (GA)      0.1623 0.1623    0.3388               2
        XGBoost (default)  0.2995 0.2995    0.6304               3
        LightGBM (default) 0.2819 0.2819    0.6315               4
7       XGBoost (GA)       0.3933 0.4522    0.8880               1
        LightGBM (GA)      0.4255 0.4931    0.9414               2
        LightGBM (default) 0.6552 0.7448    1.4582               3
        XGBoost (default)  0.7500 0.8414    1.5994               4
30      XGBoost (GA)       1.3495 1.5906    2.9116               1
        LightGBM (GA)      1.6919 1.9857    3.4217               2
        LightGBM (default) 2.2985 2.6071    4.5657               3
        XGBoost (default)  2.5414 2.8707    5.1472               4

## Ringkasan: Rata-rata MAPE (%) per Model dan Origin


In [ ]:
df_error.pivot_table(
    index=["Horizon", "Model"], columns="Origin", values="MAPE (%)", aggfunc="mean"
).style.format("{:.2f}")

## Model Terbaik per Komoditas dan Horizon (rata-rata MAPE semua origin)


In [ ]:
rata_mape = (
    df_error.groupby(["Komoditas", "Horizon", "Model"])["MAPE (%)"].mean().reset_index()
)
rata_mape.loc[
    rata_mape.groupby(["Komoditas", "Horizon"])["MAPE (%)"].idxmin()
].reset_index(drop=True)

,Komoditas,Horizon,Model,MAPE (%)
0,Bawang Merah Ukuran Sedang,1,XGBoost (GA),0.2573
1,Bawang Merah Ukuran Sedang,7,LightGBM (GA),0.6347
2,Bawang Merah Ukuran Sedang,30,LightGBM (GA),2.6834
3,Beras Kualitas Bawah I,1,LightGBM (GA),0.0000
4,Beras Kualitas Bawah I,7,LightGBM (GA),0.3680
5,Beras Kualitas Bawah I,30,XGBoost (GA),1.5553
6,Beras Kualitas Bawah II,1,XGBoost (GA),0.0002
7,Beras Kualitas Bawah II,7,LightGBM (GA),0.1160
8,Beras Kualitas Bawah II,30,LightGBM (default),0.6415
9,Beras Kualitas Medium I,1,XGBoost (GA),0.0004
